### INITIALIZATION: 
IMPORT TOOLS, SET SEED, AND CREATE INDEPENDENT TABLES

In [1]:
# libraries
import pandas as pd
import numpy as np
import uuid
from datetime import datetime, timedelta

# seed
SEED = 42
np.random.seed(SEED)


print("Libraries loaded and seed set.")

Libraries loaded and seed set.


### Independent Entities

In [2]:
# Step 2: Independent Entities (Departments)
departments_data = [
    {"name": "Traffic & Transport", "daily_capacity": 50, "vulnerability_to_storm": 5.0, "base_rate": 20},
    {"name": "Public Works", "daily_capacity": 40, "vulnerability_to_storm": 8.0, "base_rate": 15},
    {"name": "Parks & Recreation", "daily_capacity": 10, "vulnerability_to_storm": 1.5, "base_rate": 2},
    {"name": "Animal Control", "daily_capacity": 15, "vulnerability_to_storm": 1.0, "base_rate": 5},
    {"name": "Sanitation", "daily_capacity": 60, "vulnerability_to_storm": 3.0, "base_rate": 45}
]

df_departments = pd.DataFrame(departments_data)

# Give each department a unique UUID
df_departments['department_id'] = [str(uuid.uuid4()) for _ in range(len(df_departments))]

print(df_departments.head())

                  name  ...                         department_id
0  Traffic & Transport  ...  2f07659a-895b-4ee2-aeca-0eb431022f00
1         Public Works  ...  8296688e-9726-4722-8ce7-b33faf1212b4
2   Parks & Recreation  ...  7e9b838b-89d3-48d1-b8a5-99df97523a9b
3       Animal Control  ...  b71e8aec-2ad7-49b8-a65d-b487c481affd
4           Sanitation  ...  fc7577a9-e724-4f72-beff-091248ddcdd6

[5 rows x 5 columns]


In [3]:
# setup 
NUM_CITIZENS = 10000
BASE_DATE = datetime(2026, 8, 1)

# citizen ids
# generate 10000 ids for each citizen
citizen_ids = [str(uuid.uuid4()) for _ in range(NUM_CITIZENS)]

# dates
# generate random dates
random_days_ago = np.random.randint(1, 365,size=NUM_CITIZENS)
join_dates = [
    (BASE_DATE - timedelta(days=int(days))).strftime("%Y-%m-%d") 
    for days in random_days_ago
]

# clip L_civic
# np.random.normal creates the raw data without clipping 
# np.clip forces any number below 0.1 to become 0.1, and any above 2 to become 2.
raw_scores = np.random.normal(loc=0.5, scale=0.3, size=NUM_CITIZENS)
L_civics = np.clip(raw_scores, a_min=0.1, a_max=2.0)


# creation of data frame
df_citizens = pd.DataFrame({
    "citizen_id": citizen_ids,
    "join_date": join_dates,
    "L_civic": L_civics,
})

print(df_citizens.head())
print("\nL_civic:")
print(df_citizens["L_civic"].describe())

                             citizen_id   join_date   L_civic
0  28c408c0-bc5b-4a54-aa75-1ab1e92ee8a0  2026-04-20  0.798459
1  7ddfeea9-4b4a-4da5-ac9b-3a4da4180d4d  2025-08-17  0.471951
2  1bceaf64-070f-4d75-bbac-69e3829f971d  2025-11-03  1.152113
3  941c2582-bd83-48dc-94c0-9bd7d6b9ae4b  2026-04-16  0.100000
4  5428fe67-ee36-48e3-af1b-7e80b5e5deda  2026-05-21  0.211590

L_civic:
count    10000.000000
mean         0.511578
std          0.275631
min          0.100000
25%          0.296869
50%          0.499840
75%          0.701194
max          1.525584
Name: L_civic, dtype: float64


### Step 3B: Timeline & Latent Weather Shock
Simulate a 30-day calendar starting from  (August 1, 2026) and inject a hidden omitted variable  representing a decaying typhoon shock.

In [4]:
# Step 3B: Timeline & Latent Weather Shock

# 1. 30 consecutive days starting from BASE_DATE using timedelta list comprehension
# create a list of all dates from the starting base date
timeline_dates = [
    (BASE_DATE + timedelta(days=i)).strftime("%Y-%m-%d")
    for i in range(30)
]

# 2. Vectorized initialization of latent storm shock (L_storm)
# intialize a 30 row list with 0.0
L_storm = np.zeros(30)

# 3. Inject decaying typhoon shock at specific indexes (Day 15, Day 16, Day 17)
# change the values to simulate a typoon situation
L_storm[14] = 1.0  # Index 14 (Day 15): Peak typhoon shock
L_storm[15] = 0.6  # Index 15 (Day 16): Receding floodwaters
L_storm[16] = 0.2  # Index 16 (Day 17): Residual shock

# Create timeline DataFrame
df_timeline = pd.DataFrame({
    "date": timeline_dates,
    "L_storm": L_storm
})

# Verify storm pulse injection
print("Timeline created. Storm pulse slice (index 13:18):")
print(df_timeline.iloc[13:18])

Timeline created. Storm pulse slice (index 13:18):
          date  L_storm
13  2026-08-14      0.0
14  2026-08-15      1.0
15  2026-08-16      0.6
16  2026-08-17      0.2
17  2026-08-18      0.0


### Step 3C: Timeline & Department Vulnerability Interaction Grid
Perform a Cartesian cross-join between `df_timeline` (30 days) and `df_departments` (5 departments) to create a 150-row simulation grid (`df_grid`).

In [5]:
# Step 3C: Timeline & Department Vulnerability Interaction Grid

# Perform Cartesian cross-join between timeline and departments
# merge the timeline with departments table, creating 150 rows
df_grid = df_timeline.merge(df_departments, how="cross")

# Verification
print(f"Simulation grid created with shape: {df_grid.shape}")
print("\nSample rows during storm peak (2026-08-15):")
print(df_grid[df_grid["date"] == "2026-08-15"][["date", "name", "L_storm", "vulnerability_to_storm", "base_rate"]])


Simulation grid created with shape: (150, 7)

Sample rows during storm peak (2026-08-15):
          date                 name  L_storm  vulnerability_to_storm  base_rate
70  2026-08-15  Traffic & Transport      1.0                     5.0         20
71  2026-08-15         Public Works      1.0                     8.0         15
72  2026-08-15   Parks & Recreation      1.0                     1.5          2
73  2026-08-15       Animal Control      1.0                     1.0          5
74  2026-08-15           Sanitation      1.0                     3.0         45


### Step 4: Core Ticket Generation Engine
Calculates expected daily ticket volume ($\lambda$) per department using $\lambda = \text{base\_rate} + (\text{base\_rate} \times L_{\text{storm}} \times \text{vulnerability\_to\_storm})$. Samples realized ticket counts using Poisson distribution (`np.random.poisson(lam)`) and assigns tickets to citizens weighted by their civic engagement score $L_{\text{civic}}$ (hyper-reporters).

In [6]:
# Step 4: Core Ticket Generation Engine
# Goal: Simulate how many complaints each department receives daily, and who reports them.

# 1. CALCULATE EXPECTED DAILY TICKETS (lambda / average daily rate)
# Formula: BaseRate + (BaseRate * L_storm * vulnerability_to_storm)
# Why: Sunny days (L_storm=0) keep base rates (e.g. 15). Storm days (L_storm=1.0) surge rates based on vulnerability.
df_grid["lambda"] = df_grid["base_rate"] + (
    df_grid["base_rate"] * df_grid["L_storm"] * df_grid["vulnerability_to_storm"]
)

# 2. GENERATE REALIZED DAILY TICKET COUNTS (Poisson Random Sampling)
# Why Poisson? Real city complaints fluctuate around an average. np.random.poisson(lam) generates realistic random counts.
df_grid["num_tickets"] = np.random.poisson(df_grid["lambda"])

# 3. CALCULATE CITIZEN REPORTING PROBABILITIES (Hyper-Reporter Weights)
# Why: Divide each citizen's L_civic score by the sum of all scores so total probability adds up to 100% (1.0).
citizen_weights = df_citizens["L_civic"].values / df_citizens["L_civic"].sum()

# 4. ASSIGN TICKETS TO CITIZENS (Weighted Random Choice / Raffle Drum)
# Why: np.random.choice picks citizens randomly, giving higher-scoring citizens a greater chance of being picked.
ticket_records = []
for idx, row in df_grid.iterrows():
    count = row["num_tickets"]
    if count > 0:
        assigned_citizens = np.random.choice(
            df_citizens["citizen_id"].values,
            size=count,
            p=citizen_weights
        )
        for c_id in assigned_citizens:
            ticket_records.append({
                "ticket_id": str(uuid.uuid4()),      # Unique 128-bit ID per ticket
                "created_at": row["date"],          # Creation date string (YYYY-MM-DD)
                "department_id": row["department_id"], # Department receiving the ticket
                "citizen_id": c_id                  # Assigned citizen ID
            })

# 5. CONSTRUCT FINAL TICKETS DATAFRAME
df_tickets = pd.DataFrame(ticket_records)

# Verification & Summaries (Pandas Pivot Table Grouping)
print(f"Total synthetic tickets generated: {len(df_tickets)}")
print("\nHead of df_tickets (First 5 rows):")
print(df_tickets.head())
print("\nDaily ticket counts during storm surge (2026-08-13 to 2026-08-18):")
daily_summary = df_grid.groupby("date")["num_tickets"].sum().reset_index()
print(daily_summary.iloc[12:18])


Total synthetic tickets generated: 3199

Head of df_tickets (First 5 rows):
                              ticket_id  ...                            citizen_id
0  beb0532d-825a-48da-97a5-ef1ea21395f8  ...  da77df31-1cee-43f0-b0cc-0ae4dc3c851c
1  bcb7b694-7f23-41d5-8f04-daba56cfdb4c  ...  64590123-4461-43c6-af79-4a7cad16137c
2  4f9cb6f9-2b5d-4a1b-9c5b-cdb97acf685c  ...  e39ac1ef-52e8-4c3c-b15c-eb3c0969ac64
3  f5e663c7-0b79-400f-9eb3-2162a4d952e6  ...  95b8d9b5-a66f-409a-ac39-eb741df064de
4  9ad605de-d258-4bda-a932-a761823e18e3  ...  5f99a26d-f1de-4325-bc6e-8de68d415a98

[5 rows x 4 columns]

Daily ticket counts during storm surge (2026-08-13 to 2026-08-18):
          date  num_tickets
12  2026-08-13           87
13  2026-08-14           65
14  2026-08-15          491
15  2026-08-16          275
16  2026-08-17          141
17  2026-08-18           86
